In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Input dan output primitif

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** Keluaran beta bagi model pelaksanaan baharu kini tersedia. Model pelaksanaan terarah memberikan lebih fleksibiliti semasa menyesuaikan aliran kerja pengurangan ralat anda. Lihat panduan [Model pelaksanaan terarah](/guides/directed-execution-model) untuk maklumat lanjut.


<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.4.0
```
</details>
Halaman ini memberikan gambaran keseluruhan tentang input dan output primitif Qiskit Runtime yang melaksanakan beban kerja pada sumber pengkomputeran IBM Quantum&reg;. Primitif ini membolehkan anda mentakrifkan beban kerja yang divektorkan dengan cekap menggunakan struktur data yang dikenali sebagai **Primitive Unified Bloc (PUB)**. PUB ini merupakan unit kerja asas yang diperlukan oleh QPU untuk melaksanakan beban kerja tersebut. Ia digunakan sebagai input kepada kaedah [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) untuk primitif Sampler dan Estimator, yang melaksanakan beban kerja yang ditakrifkan sebagai satu tugas. Kemudian, selepas tugas selesai, hasilnya dikembalikan dalam format yang bergantung pada PUB yang digunakan serta pilihan masa jalan yang ditetapkan daripada primitif Sampler atau Estimator.
<span id="pubs"></span>
## Gambaran keseluruhan PUB
Apabila menggunakan kaedah [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) sesebuah primitif, argumen utama yang diperlukan ialah satu `list` yang mengandungi satu atau lebih tupel — satu bagi setiap Circuit yang dilaksanakan oleh primitif tersebut. Setiap tupel ini dianggap sebagai PUB, dan elemen yang diperlukan bagi setiap tupel dalam senarai bergantung pada primitif yang digunakan. Data yang diberikan kepada tupel ini juga boleh disusun dalam pelbagai bentuk untuk memberikan fleksibiliti dalam beban kerja melalui penyiaran — peraturannya diterangkan dalam [bahagian berikut](#broadcasting-rules).

### Estimator PUB
Untuk primitif Estimator, format PUB mestilah mengandungi paling banyak empat nilai:
- Satu `QuantumCircuit` tunggal, yang mungkin mengandungi satu atau lebih objek [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- Senarai satu atau lebih boleh cerap (observables), yang menentukan nilai jangkaan untuk dianggarkan, disusun dalam suatu tatasusunan (contohnya, satu boleh cerap tunggal diwakili sebagai tatasusunan 0-d, senarai boleh cerap sebagai tatasusunan 1-d, dan seterusnya). Data boleh berada dalam mana-mana satu format `ObservablesArrayLike` seperti `Pauli`, `SparsePauliOp`, `PauliList`, atau `str`.
   > **Note:** Jika anda mempunyai dua boleh cerap yang bertukar-tukar (commuting) dalam PUB yang berbeza tetapi dengan Circuit yang sama, ia tidak akan dianggarkan menggunakan pengukuran yang sama. Setiap PUB mewakili asas pengukuran yang berbeza, justeru pengukuran berasingan diperlukan untuk setiap PUB. Untuk memastikan boleh cerap yang bertukar-tukar dianggarkan menggunakan pengukuran yang sama, ia mesti dikumpulkan dalam PUB yang sama.
- Koleksi nilai parameter untuk mengikat Circuit. Ini boleh ditetapkan sebagai satu objek seperti tatasusunan di mana indeks terakhir adalah atas objek `Parameter` Circuit, atau diabaikan (atau setara, ditetapkan kepada `None`) jika Circuit tidak mempunyai objek `Parameter`.
- (Pilihan) ketepatan sasaran untuk nilai jangkaan yang hendak dianggarkan

### Sampler PUB
Untuk primitif Sampler, format tupel PUB mengandungi paling banyak tiga nilai:
- Satu `QuantumCircuit` tunggal, yang mungkin mengandungi satu atau lebih objek [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
   *Nota: Circuit ini juga seharusnya mengandungi arahan pengukuran untuk setiap Qubit yang hendak dipersamplkan.*
- Koleksi nilai parameter untuk mengikat Circuit terhadap $\theta_k$ (hanya diperlukan jika ada objek `Parameter` yang mesti diikat semasa masa jalan)
- (Pilihan) bilangan tembakan untuk mengukur Circuit
---

Kod berikut menunjukkan contoh set input yang divektorkan kepada primitif `Estimator` dan melaksanakannya pada Backend IBM&reg; sebagai satu objek `RuntimeJobV2 `.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray
from qiskit.primitives import StatevectorEstimator


import numpy as np

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit without providing a backend
pm = generate_preset_pass_manager(optimization_level=1)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 10),
        np.linspace(-4 * np.pi, 4 * np.pi, 10),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator = StatevectorEstimator()
estimator_pub = (transpiled_circuit, observables, params)

# Run the transpiled circuit
# using the set of parameters and observables.

job = estimator.run([estimator_pub])
result = job.result()

<span id="broadcasting"></span>
### Broadcasting rules

The PUBs aggregate elements from multiple arrays (observables and parameter values) by following the same broadcasting rules as NumPy. This section briefly summarizes those rules.  For a detailed explanation, see the [NumPy broadcasting rules documentation](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Rules:

* Input arrays do not need to have the same number of dimensions.
  * The resulting array will have the same number of dimensions as the input array with the largest dimension.
  * The size of each dimension is the largest size of the corresponding dimension.
  * Missing dimensions are assumed to have size one.
* Shape comparisons start with the rightmost dimension and continue to the left.
* Two dimensions are compatible if their sizes are equal or if one of them is 1.

Examples of array pairs that broadcast:

```text
A1     (1d array):      1
A2     (2d array):  3 x 5
Result (2d array):  3 x 5


A1     (3d array):  11 x 2 x 7
A2     (3d array):  11 x 1 x 7
Result (3d array):  11 x 2 x 7
```

Examples of array pairs that do not broadcast:

```text
A1     (1d array):  5
A2     (1d array):  3

A1     (2d array):      2 x 1
A2     (3d array):  6 x 5 x 4 # This would work if the middle dimension were 2, but it is 5.
```

`Estimator` returns one expectation value estimate for each element of the broadcasted shape.

Here are some examples of common patterns expressed in terms of array broadcasting.  Their accompanying visual representation is shown in the figure that follows:


Parameter value sets are represented by n x m arrays, and observable arrays are represented by one or more single-column arrays. For each example in the previous code, the parameter value sets are combined with their observable array to create the resulting expectation value estimates.

 - *Example 1*: (broadcast single observable) has a parameter value set that is a 5x1 array and a 1x1 observables array.  The one item in the observables array is combined with each item in the parameter value set to create a single 5x1 array where each item is a combination of the original item in the parameter value set with the item in the observables array.

 - *Example 2*: (zip) has a 5x1 parameter value set and a 5x1 observables array.  The output is a 5x1 array where each item is a combination of the nth item in the parameter value set with the nth item in the observables array.

  - *Example 3*: (outer/product) has a 1x6 parameter value set and a 4x1 observables array.  Their combination results in a 4x6 array that is created by combining each item in the parameter value set with *every* item in the observables array, and thus each parameter value becomes an entire column in the output.

  - *Example 4*: (Standard nd generalization) has a 3x6 parameter value set array and two 3x1 observables array.  These combine to create two 3x6 output arrays in a similar manner to the previous example.

![This image illustrates several visual representations of array broadcasting.](../docs/images/guides/primitive-input-output/broadcasting.svg "Visual representation of broadcasting")

In [2]:
# Broadcast single observable
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = SparsePauliOp("ZZZ")  # shape ()
# >> pub result has shape (5,)

# Zip
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = [
    SparsePauliOp(pauli) for pauli in ["III", "XXX", "YYY", "ZZZ", "XYZ"]
]  # shape (5,)
# >> pub result has shape (5,)

# Outer/Product
parameter_values = np.random.uniform(size=(1, 6))  # shape (1, 6)
observables = [
    [SparsePauliOp(pauli)] for pauli in ["III", "XXX", "YYY", "ZZZ"]
]  # shape (4, 1)
# >> pub result has shape (4, 6)

# Standard nd generalization
parameter_values = np.random.uniform(size=(3, 6))  # shape (3, 6)
observables = [
    [
        [SparsePauliOp(["XII"])],
        [SparsePauliOp(["IXI"])],
        [SparsePauliOp(["IIX"])],
    ],
    [
        [SparsePauliOp(["ZII"])],
        [SparsePauliOp(["IZI"])],
        [SparsePauliOp(["IIZ"])],
    ],
]  # shape (2, 3, 1)
# >> pub result has shape (2, 3, 6)

### Peraturan penyiaran
PUB menggabungkan elemen daripada pelbagai tatasusunan (boleh cerap dan nilai parameter) dengan mengikuti peraturan penyiaran yang sama seperti NumPy. Bahagian ini meringkaskan peraturan tersebut secara ringkas. Untuk penjelasan terperinci, lihat [dokumentasi peraturan penyiaran NumPy](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Peraturan:

* Tatasusunan input tidak perlu mempunyai bilangan dimensi yang sama.
  * Tatasusunan yang terhasil akan mempunyai bilangan dimensi yang sama dengan tatasusunan input yang mempunyai dimensi terbesar.
  * Saiz setiap dimensi ialah saiz terbesar bagi dimensi yang sepadan.
  * Dimensi yang tiada dianggap mempunyai saiz satu.
* Perbandingan bentuk bermula dari dimensi paling kanan dan diteruskan ke kiri.
* Dua dimensi serasi jika saiznya sama atau jika salah satunya ialah 1.

Contoh pasangan tatasusunan yang boleh disiarkan:

In [3]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10)), metadata={'target_precision': 0.0, 'circuit_metadata': {}})], metadata={'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10))

And this DataBin has attributes: dict_keys(['evs', 'stds'])
Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). 

The expectation values measured from this PUB are: 
[[ 3.06161700e-16  4.52395120e-01  4.36594428e-01  2.16506351e-01
   6.33718361e-01 -6.33718361e-01 -2.16506351e-01 -4.36594428e-01
  -4.52395120e-01 -3.06161700e-16]
 [ 1.22464680

Contoh pasangan tatasusunan yang tidak boleh disiarkan:

In [4]:
from qiskit.primitives import StatevectorSampler

# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

sampler = StatevectorSampler()

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=1024, num_bits=10>)

The shape of register `meas` is (1024, 2).

The bytes in register `alpha`, shot by shot:
[[  0   0]
 [  3 255]
 [  0   0]
 ...
 [  3 255]
 [  3 255]
 [  3 255]]



`EstimatorV2` mengembalikan satu anggaran nilai jangkaan bagi setiap elemen bentuk yang disiarkan.

Berikut adalah beberapa contoh corak biasa yang dinyatakan dari segi penyiaran tatasusunan. Representasi visual yang menyertainya ditunjukkan dalam rajah yang berikut:

Set nilai parameter diwakili oleh tatasusunan n x m, dan tatasusunan boleh cerap diwakili oleh satu atau lebih tatasusunan lajur tunggal. Bagi setiap contoh dalam kod sebelumnya, set nilai parameter digabungkan dengan tatasusunan boleh cerap mereka untuk menghasilkan anggaran nilai jangkaan.

 - *Contoh 1*: (siarkan satu boleh cerap) mempunyai set nilai parameter yang merupakan tatasusunan 5x1 dan tatasusunan boleh cerap 1x1. Item tunggal dalam tatasusunan boleh cerap digabungkan dengan setiap item dalam set nilai parameter untuk menghasilkan tatasusunan 5x1 tunggal di mana setiap item adalah gabungan item asal dalam set nilai parameter dengan item dalam tatasusunan boleh cerap.

 - *Contoh 2*: (zip) mempunyai set nilai parameter 5x1 dan tatasusunan boleh cerap 5x1. Output ialah tatasusunan 5x1 di mana setiap item adalah gabungan item ke-n dalam set nilai parameter dengan item ke-n dalam tatasusunan boleh cerap.

  - *Contoh 3*: (luar/hasil) mempunyai set nilai parameter 1x6 dan tatasusunan boleh cerap 4x1. Gabungan mereka menghasilkan tatasusunan 4x6 yang dicipta dengan menggabungkan setiap item dalam set nilai parameter dengan *setiap* item dalam tatasusunan boleh cerap, justeru setiap nilai parameter menjadi satu lajur penuh dalam output.

  - *Contoh 4*: (Generalisasi nd piawai) mempunyai tatasusunan set nilai parameter 3x6 dan dua tatasusunan boleh cerap 3x1. Ini digabungkan untuk menghasilkan dua tatasusunan output 3x6 dengan cara yang serupa dengan contoh sebelumnya.

![Imej ini menggambarkan beberapa representasi visual penyiaran tatasusunan](../docs/images/guides/primitive-input-output/broadcasting.svg "Representasi visual penyiaran")

In [5]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'0000000000': 492, '1111111111': 532}


<Admonition type="tip" title="SparsePauliOp">
Setiap `SparsePauliOp` dikira sebagai satu elemen dalam konteks ini, tanpa mengira bilangan Pauli yang terkandung dalam `SparsePauliOp`. Oleh itu, bagi tujuan peraturan penyiaran ini, semua elemen berikut mempunyai bentuk yang sama:

In [6]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=1024, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=1024, num_bits=9>)


Senarai pengoperasi berikut, walaupun setara dari segi maklumat yang terkandung, mempunyai bentuk yang berbeza:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object,
        |       |     ## which includes data such as the following:
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object,
        |       |     ## which includes data such as the following:
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```
</Admonition>
## Gambaran keseluruhan output primitif
Setelah satu atau lebih PUB dihantar ke QPU untuk dilaksanakan dan tugas berjaya diselesaikan, data dikembalikan sebagai objek kontena [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) yang diakses dengan memanggil kaedah `RuntimeJobV2.result()`. `PrimitiveResult` mengandungi senarai boleh ulang bagi objek [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) yang mengandungi hasil pelaksanaan bagi setiap PUB. Bergantung pada primitif yang digunakan, data ini akan berbentuk nilai jangkaan dan bar ralat dalam kes Estimator, atau sampel output Circuit dalam kes Sampler.

Setiap elemen dalam senarai ini sepadan dengan setiap PUB yang diserahkan kepada kaedah `run()` primitif (contohnya, tugas yang diserahkan dengan 20 PUB akan mengembalikan objek `PrimitiveResult` yang mengandungi senarai 20 `PubResult`, satu sepadan dengan setiap PUB).

Setiap objek `PubResult` ini mempunyai atribut `data` dan `metadata`. Atribut `data` ialah [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) yang disesuaikan dan mengandungi nilai pengukuran sebenar, sisihan piawai, dan sebagainya. `DataBin` ini mempunyai pelbagai atribut bergantung pada bentuk atau struktur PUB yang berkaitan serta pilihan pengurangan ralat yang ditetapkan oleh primitif yang digunakan untuk menghantar tugas (contohnya, [ZNE](./error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) atau [PEC](./error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)). Sementara itu, atribut `metadata` mengandungi maklumat tentang pilihan masa jalan dan pengurangan ralat yang digunakan (diterangkan kemudian dalam bahagian [Metadata hasil](#result-metadata) pada halaman ini).

Berikut ialah garis besar visual struktur data `PrimitiveResult`:

<Tabs>
    <TabItem value="estimator" label="Output Estimator">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data for first PUB (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second PUB
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
    <TabItem value="sampler" label="Output Sampler">

In [7]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (1024, 1).
The bytes in register `alpha`, shot by shot:
[[1]
 [1]
 [1]
 ...
 [0]
 [0]
 [1]]

The shape of register `beta` is (1024, 2).
The bytes in register `beta`, shot by shot:
[[  1 255]
 [  1 255]
 [  1 255]
 ...
 [  0   0]
 [  0   0]
 [  1 255]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (1024, 1).
The bytes in `beta` after bit-wise slicing:
[[7]
 [7]
 [7]
 ...
 [0]
 [0]
 [7]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  1 255]
 [  1 255]
 [  1 255]
 [  1 255]
 [  1 255]]



Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: -0.017578125
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: -0.017578125

The shape of the merged results is (1024, 2).
The bytes of the merged results:
[[  3 255]
 [  3 255]
 [  3 255]
 ...
 [  0   0]
 [  0   0]
 [  3 255]]



</TabItem>
</Tabs>

Ringkasnya, satu tugas mengembalikan objek `PrimitiveResult` dan mengandungi senarai satu atau lebih objek `PubResult`. Objek `PubResult` ini kemudian menyimpan data pengukuran bagi setiap PUB yang diserahkan kepada tugas tersebut.

Setiap `PubResult` mempunyai format dan atribut yang berbeza berdasarkan jenis primitif yang digunakan untuk tugas tersebut. Perinciannya diterangkan di bawah.

### Output Estimator
Setiap `PubResult` untuk primitif Estimator mengandungi sekurang-kurangnya satu tatasusunan nilai jangkaan (`PubResult.data.evs`) dan sisihan piawai yang berkaitan (sama ada `PubResult.data.stds` atau `PubResult.data.ensemble_standard_error` bergantung pada `resilience_level` yang digunakan), tetapi boleh mengandungi lebih banyak data bergantung pada pilihan pengurangan ralat yang ditetapkan.

Petikan kod di bawah menerangkan format `PrimitiveResult` (dan `PubResult` yang berkaitan) bagi tugas yang dibuat di atas.

In [8]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'version' : 2,

The metadata of the PubResult result is:
'shots' : 1024,
'circuit_metadata' : {},


## Next steps

<Admonition type="tip" title="Recommendations">

  -  Review the [Qiskit primitives](/docs/api/qiskit/primitives) API.
  -  Review the [Qiskit Aer primitives](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) API.
  -  Learn more about the [Qiskit Runtime primitives](/docs/guides/qiskit-runtime-primitives).
  -  Review the [Qiskit Runtime Estimator](/docs/api/qiskit-ibm-runtime/estimator-v2) API.
  -  Review the [Qiskit Runtime Sampler](/docs/api/qiskit-ibm-runtime/sampler-v2) API.
</Admonition>